# S3. Reading a front and its quality indicators, in Python

The companion of VAMOS-Studio's tutorial *S3*: fronts of different shapes, how quality indicators respond to an approximation, and the indicators of a run. It pairs with VAMOS's tutorial [T07. Quality and budget](https://vamos-optimization.org/tutorials/t07-quality-and-budget/).

In [ ]:
from importlib.resources import files

import numpy as np
import plotly.graph_objects as go
import vamos
from vamos.algorithms import NSGAIIConfig
from vamos.foundation.quality_indicators import get_indicator

print("VAMOS", vamos.__version__)


def reference_front(name: str) -> np.ndarray:
    """A reference front copied from jMetal, as VAMOS-Studio ships it (e.g. "ZDT1.csv")."""
    path = files("vamos_studio") / "resources" / "referenceFronts" / name
    return np.loadtxt(path, delimiter=",", ndmin=2)


def indicators(front: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    """HV and IGD+ as VAMOS-Studio computes them: objectives normalized by the reference front."""
    lower = reference.min(axis=0)
    span = reference.max(axis=0) - lower
    span[span == 0] = 1.0
    normalized, normalized_reference = (front - lower) / span, (reference - lower) / span
    hv = get_indicator("hv", reference_point=np.ones(front.shape[1]))
    igd_plus = get_indicator("igd_plus", reference_front=normalized_reference)
    return {
        "HV": round(float(hv.compute(np.clip(normalized, 0.0, 1.0)).value), 4),
        "IGD+": round(float(igd_plus.compute(normalized).value), 4),
    }


def plot(front: np.ndarray, reference: np.ndarray, title: str) -> go.Figure:
    """A two-objective front in blue over its reference front in grey."""
    figure = go.Figure()
    figure.add_scatter(
        x=reference[:, 0],
        y=reference[:, 1],
        mode="markers",
        name="Reference front",
        marker={"size": 3, "color": "#b7bcc5"},
    )
    figure.add_scatter(
        x=front[:, 0],
        y=front[:, 1],
        mode="markers",
        name="Front",
        marker={"size": 7, "color": "#1769aa"},
    )
    figure.update_layout(title=title, xaxis_title="f1", yaxis_title="f2")
    return figure

## Shapes of fronts

Convex, concave and disconnected:

In [ ]:
for name in ("ZDT1", "ZDT2", "ZDT3"):
    front = reference_front(f"{name}.csv")
    plot(front, front, f"The Pareto front of {name}").show()

## Measuring an approximation

Take evenly spread points of the front of ZDT1 and move them by a fraction of each objective's range, as the tutorial's sliders do. Few points lower HV (spread); a shift worsens every indicator (convergence).

In [ ]:
zdt1 = reference_front("ZDT1.csv")
zdt1 = zdt1[np.argsort(zdt1[:, 0])]


def approximation(points: int, shift: float) -> np.ndarray:
    """`points` evenly spread points of the front, moved by `shift` times each range."""
    rows = np.unique(np.round(np.linspace(0, len(zdt1) - 1, points)).astype(int))
    return zdt1[rows] + shift * (zdt1.max(axis=0) - zdt1.min(axis=0))


for points, shift in ((100, 0.0), (5, 0.0), (100, 0.05)):
    print(points, "points, shift", shift, indicators(approximation(points, shift), zdt1))

## The indicators of a run

NSGA-II on DTLZ2 with three objectives: the points lie on the sphere, but are spread unevenly.

In [ ]:
default = NSGAIIConfig.default(pop_size=100, n_var=12)
result = vamos.optimize(
    "dtlz2",
    algorithm="nsgaii",
    algorithm_config=default,
    n_obj=3,
    max_evaluations=25000,
    seed=1,
)
front = np.asarray(result.F)
dtlz2 = reference_front("DTLZ2.3D.csv")
print(len(front), "solutions", indicators(front, dtlz2))  # 100 solutions, HV 0.3745, IGD+ 0.039
figure = go.Figure()
figure.add_scatter3d(
    x=dtlz2[:, 0],
    y=dtlz2[:, 1],
    z=dtlz2[:, 2],
    mode="markers",
    name="Reference front",
    marker={"size": 1, "color": "#b7bcc5"},
)
figure.add_scatter3d(
    x=front[:, 0],
    y=front[:, 1],
    z=front[:, 2],
    mode="markers",
    name="Front",
    marker={"size": 3, "color": "#1769aa"},
)
figure.show()